# Social Media Engagement Analysis: Data Cleaning & Preprocessing Pipeline
**College Data Visualization (DV) Project**  
**Task Phase**: Milestone 2 - Data Cleaning & Preprocessing  
**Author**: Data Analytics & Visualization Team  
**Objective**: Transform raw social media engagement data into an analysis-ready, clean dataset following disciplined data science methodologies. All decisions, treatments, and transformations are documented and reproducible.


## 1. Environment Setup & Raw Data Ingestion
We load the raw dataset from `data/social_media_sentiment_raw.csv` using relative file paths so this pipeline runs reproducibly on any operating system.
The original raw dataset is preserved unchanged in its raw location.


In [1]:
import os
import pandas as pd
import numpy as np

# Cross-platform relative path resolution
raw_csv_path = os.path.join('data', 'social_media_sentiment_raw.csv') if os.path.exists(os.path.join('data', 'social_media_sentiment_raw.csv')) else os.path.join('..', 'data', 'social_media_sentiment_raw.csv')
processed_dir = os.path.join('data', 'processed') if os.path.exists('data') else os.path.join('..', 'data', 'processed')
processed_csv_path = os.path.join(processed_dir, 'social_media_engagement_cleaned.csv')

print(f"Loading raw dataset from: {raw_csv_path}")
df_raw = pd.read_csv(raw_csv_path)

print(f"Initial raw dataset shape: {df_raw.shape[0]:,} rows and {df_raw.shape[1]} columns")
print(f"Preserving original CSV file immutability: verified.")


Loading raw dataset from: data\social_media_sentiment_raw.csv
Initial raw dataset shape: 2,260 rows and 13 columns
Preserving original CSV file immutability: verified.


## 2. Categorical Standardization (Platforms and Countries)
### Issues Addressed:
1. **Platform Casing**: Raw data contained 10 distinct values due to uppercase variants (e.g., `TWITTER`, `INSTAGRAM`, `TIKTOK`, `FACEBOOK`, `LINKEDIN`). We map these to canonical brand names (`Twitter`, `Instagram`, `TikTok`, `Facebook`, `LinkedIn`).
2. **Country Whitespace & Casing**: Raw data contained trailing spaces and lowercase variants (e.g., `'usa  '`, `'india  '`). We strip whitespace and standardize names while preserving recognized country acronyms (`USA`, `UK`, `UAE`).
3. **Missing Countries**: Per project requirements, missing countries are explicitly labeled as `'Unknown'` for non-geographic analysis, allowing records to be preserved without inventing geography.


In [2]:
# Work on a distinct copy to ensure raw data integrity
df = df_raw.copy()

# 1. Standardize platform names with canonical brand casing
platform_map = {
    'twitter': 'Twitter',
    'instagram': 'Instagram',
    'tiktok': 'TikTok',
    'facebook': 'Facebook',
    'linkedin': 'LinkedIn'
}
df['platform'] = df['platform'].astype(str).str.strip().str.lower().map(platform_map)

# 2. Standardize country names, preserve acronyms, and assign 'Unknown' to missing
country_acronyms = {'usa': 'USA', 'uk': 'UK', 'uae': 'UAE'}

def clean_country(val):
    if pd.isna(val) or str(val).strip().lower() in ['', 'nan', 'none', 'unknown']:
        return 'Unknown'
    cleaned = str(val).strip()
    if cleaned.lower() in country_acronyms:
        return country_acronyms[cleaned.lower()]
    return cleaned.title()

df['country'] = df['country'].apply(clean_country)

print("Standardized Platform Distribution:")
print(df['platform'].value_counts())

print("\nStandardized Country Distribution:")
print(df['country'].value_counts())


Standardized Platform Distribution:
platform
Twitter      745
Instagram    614
TikTok       398
Facebook     379
LinkedIn     124
Name: count, dtype: int64

Standardized Country Distribution:
country
USA             466
India           389
UK              217
Canada          176
Australia       164
Germany         149
Unknown         126
UAE             117
France          108
Brazil          107
Japan           106
South Africa     82
Mexico           53
Name: count, dtype: int64


## 3. Follower Count Formatting & Duplicate Post Investigation
### Issues Addressed:
1. **Followers Count Formatting**: 67 entries contained comma separators (e.g., `'20,454'`), forcing pandas to interpret the column as `object`. We remove commas and convert to numeric values.
2. **Duplicate `post_id` Resolution**:
   - In raw data, 60 duplicate instances of `post_id` existed.
   - 44 were character-for-character identical across all columns.
   - The remaining 16 duplicate pairs differed only in formatting artifacts (platform casing or comma separators in followers).
   - Once formatting is normalized, all 60 pairs represent identical post records.
   - **Documented Deduplication Rule**: Retain the first valid occurrence of each `post_id` and remove the 60 redundant duplicate rows, yielding 2,200 unique post records.


In [3]:
# 1. Strip commas and convert followers_count to numeric
df['followers_count'] = pd.to_numeric(
    df['followers_count'].astype(str).str.replace(',', '').str.strip(),
    errors='coerce'
)

# 2. Investigate duplicates prior to deduplication
pre_exact_dups = df.duplicated().sum()
pre_id_dups = df.duplicated(subset=['post_id']).sum()
print(f"Exact full-row duplicates after normalization: {pre_exact_dups}")
print(f"Duplicate post_ids after normalization: {pre_id_dups}")

# 3. Apply documented deduplication rule: retain first valid record per unique post_id
rows_before = len(df)
df = df.drop_duplicates(subset=['post_id'], keep='first').copy()
rows_after = len(df)
removed_duplicates = rows_before - rows_after

print(f"\nDeduplication complete:")
print(f"  Rows before deduplication: {rows_before:,}")
print(f"  Rows after deduplication:  {rows_after:,}")
print(f"  Redundant duplicates removed: {removed_duplicates}")
assert df['post_id'].duplicated().sum() == 0, "Assertion failed: Duplicate post_ids still exist!"
print("Assertion verified: 0 duplicate post_ids remaining.")


Exact full-row duplicates after normalization: 60
Duplicate post_ids after normalization: 60

Deduplication complete:
  Rows before deduplication: 2,260
  Rows after deduplication:  2,200
  Redundant duplicates removed: 60
Assertion verified: 0 duplicate post_ids remaining.


## 4. Temporal Standardization & Feature Engineering
### Transformations:
1. Convert `timestamp` to `datetime64[ns]`.
2. Extract calendar and cyclic temporal features:
   - `year`: Calendar year (2023 or 2024).
   - `month`: Numerical month (1 to 12).
   - `month_year`: Period string `YYYY-MM` for longitudinal trend plotting.
   - `day_of_week`: Day name (Monday through Sunday) for posting pattern analysis.
   - `hour`: Hour of posting (0 to 23) for diurnal heatmap visualizations.


In [4]:
# Convert timestamp to datetime
df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
assert df['timestamp'].isna().sum() == 0, "Assertion failed: Found unparseable timestamps!"

# Extract derived temporal features
df['year'] = df['timestamp'].dt.year.astype('int32')
df['month'] = df['timestamp'].dt.month.astype('int32')
df['month_year'] = df['timestamp'].dt.to_period('M').astype(str)
df['day_of_week'] = df['timestamp'].dt.day_name()
df['hour'] = df['timestamp'].dt.hour.astype('int32')

print("Temporal Range:")
print(f"  Earliest post: {df['timestamp'].min()}")
print(f"  Latest post:   {df['timestamp'].max()}")
print("\nPosts per calendar year:")
print(df['year'].value_counts().sort_index())


Temporal Range:
  Earliest post: 2023-01-01 01:08:09
  Latest post:   2024-12-31 21:11:13

Posts per calendar year:
year
2023    1091
2024    1109
Name: count, dtype: int64


## 5. Engagement Metrics Cleaning & Sentinel Value Treatment
### Issues Addressed:
1. **Negative Likes (`likes = -1`)**: 8 records contained `-1` in the raw dataset. Because a negative like count is physically impossible in social media platforms, this is recognized as an ingestion error or sentinel code for missing data. We convert `likes = -1` to missing (`NaN`) and document the decision.
2. **Preserving Missingness**:
   - We do **not** assume missing comments mean 0 comments; missing values in `comments` (44 records in deduplicated data) are strictly preserved as missing.
   - We do **not** arbitrarily fill missing values in `followers_count` (88 missing), `verified` (44 missing), `hashtags` (762 missing), or `sentiment` (33 missing).
3. **Data Type Casting**: Count metrics (`likes`, `retweets_shares`, `comments`, `followers_count`) are cast to pandas nullable integer `Int64` so that true integers are displayed without decimal artifacts while naturally preserving missing values as `<NA>`.


In [5]:
# 1. Identify and convert likes = -1 to missing (NaN)
neg_likes_mask = df['likes'] < 0
neg_likes_count = neg_likes_mask.sum()
print(f"Negative likes count before treatment: {neg_likes_count}")

# Replace negative sentinel values with NaN
df.loc[neg_likes_mask, 'likes'] = np.nan
print("Decision documented: Converted 8 invalid likes (-1) to NaN.")

# 2. Cast numeric counts to nullable integer type Int64
df['followers_count'] = df['followers_count'].round().astype('Int64')
df['likes'] = df['likes'].astype('Int64')
df['retweets_shares'] = df['retweets_shares'].astype('Int64')
df['comments'] = df['comments'].astype('Int64')

print("\nSummary of Engagement Numeric Types:")
print(df[['followers_count', 'likes', 'retweets_shares', 'comments']].dtypes)


Negative likes count before treatment: 8
Decision documented: Converted 8 invalid likes (-1) to NaN.

Summary of Engagement Numeric Types:
followers_count    Int64
likes              Int64
retweets_shares    Int64
comments           Int64
dtype: object


## 6. Derived Engagement Metrics & Analytical Formulations
### Derived Metrics:
1. **Total Engagement**:
   - Formula: $\text{total\_engagement} = \text{likes} + \text{retweets\_shares} + \text{comments}$
   - Policy: Because missing comments and missing likes represent unrecorded figures, `total_engagement` evaluates to missing (`<NA>`) when any constituent engagement metric is missing.
2. **Engagement Rate per Follower (%)**:
   - Formula: $\text{engagement\_rate\_per\_follower} = \left(\frac{\text{total\_engagement}}{\text{followers\_count}}\right) \times 100$
   - Calculation Scope: Evaluated **strictly** where `followers_count` is valid and $> 0$, and `total_engagement` is valid. Otherwise set to `NaN`.
   - **Documented Limitations**:
     1. **Virality & Discovery**: Social algorithms (e.g., TikTok FYP, Twitter/X For You, Instagram Explore) distribute posts far beyond an author's follower base, leading to engagement rates exceeding 100% on breakout posts.
     2. **Impression Data Absence**: True industry-standard engagement rate is calculated as $(\text{Engagements} / \text{Impressions}) \times 100$. Since views/impressions are unavailable in this schema, follower count serves as an imperfect proxy for account scale, not actual post reach.
     3. **Small-Denominator Sensitivity**: Micro-accounts with very few followers (e.g., 20 followers) generate disproportionately inflated engagement rates with only modest interaction counts.


In [6]:
# 1. Total Engagement (strict sum preserving missingness)
df['total_engagement'] = df['likes'] + df['retweets_shares'] + df['comments']

# 2. Engagement Rate per Follower (%)
valid_rate_condition = (
    df['followers_count'].notna() &
    (df['followers_count'] > 0) &
    df['total_engagement'].notna()
)

df['engagement_rate_per_follower'] = np.nan
df.loc[valid_rate_condition, 'engagement_rate_per_follower'] = (
    df.loc[valid_rate_condition, 'total_engagement'] / df.loc[valid_rate_condition, 'followers_count'].astype(float)
) * 100

print("Derived Metrics Overview:")
print(f"Total Engagement valid count: {df['total_engagement'].notna().sum():,} | Missing: {df['total_engagement'].isna().sum()}")
print(f"Engagement Rate valid count:  {df['engagement_rate_per_follower'].notna().sum():,} | Missing: {df['engagement_rate_per_follower'].isna().sum()}")

print("\nDescriptive Statistics of Engagement Metrics:")
print(df[['likes', 'retweets_shares', 'comments', 'total_engagement', 'engagement_rate_per_follower']].describe().to_string())


Derived Metrics Overview:
Total Engagement valid count: 2,148 | Missing: 52
Engagement Rate valid count:  2,064 | Missing: 136

Descriptive Statistics of Engagement Metrics:
           likes  retweets_shares  comments  total_engagement  engagement_rate_per_follower
count     2192.0           2200.0    2156.0            2148.0                   2064.000000
mean   36.088504             6.62  3.450835         46.131285                      6.622331
std    16.410481         4.707657   2.73601          21.74807                     15.127765
min          5.0              0.0       0.0               5.0                      0.016633
25%         25.0              3.0      1.75              31.0                      0.839908
50%         33.0              6.0       3.0              42.0                      2.342568
75%         44.0              9.0       5.0              56.0                      6.525196
max        126.0             37.0      20.0             169.0                    250.00000

## 7. Outlier Investigation
We audit extreme values using the standard Interquartile Range (IQR) fence method ($Q_3 + 1.5 \times \text{IQR}$).
Per guidelines, **no unusually high engagement values are deleted**, as viral posts and high-performing content represent genuine social media behavior rather than corrupted data.


In [7]:
print("=== ENGAGEMENT OUTLIER AUDIT (IQR METHOD) ===")
metrics_to_check = ['likes', 'retweets_shares', 'comments', 'total_engagement', 'engagement_rate_per_follower']

for metric in metrics_to_check:
    s = df[metric].dropna()
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    upper_fence = q3 + 1.5 * iqr
    outliers = s[s > upper_fence]
    print(f"{metric:30s} | Q1: {q1:6.2f} | Median: {s.median():6.2f} | Q3: {q3:6.2f} | Upper Fence: {upper_fence:6.2f} | Outliers: {len(outliers):3d} ({len(outliers)/len(s)*100:4.1f}%) | Max: {s.max():6.2f}")

print("\nOutlier Investigation Decision:")
print("All high-engagement outliers are preserved. They represent authentic viral content typical of social networks.")


=== ENGAGEMENT OUTLIER AUDIT (IQR METHOD) ===
likes                          | Q1:  25.00 | Median:  33.00 | Q3:  44.00 | Upper Fence:  72.50 | Outliers:  77 ( 3.5%) | Max: 126.00
retweets_shares                | Q1:   3.00 | Median:   6.00 | Q3:   9.00 | Upper Fence:  18.00 | Outliers:  49 ( 2.2%) | Max:  37.00
comments                       | Q1:   1.75 | Median:   3.00 | Q3:   5.00 | Upper Fence:   9.88 | Outliers:  74 ( 3.4%) | Max:  20.00
total_engagement               | Q1:  31.00 | Median:  42.00 | Q3:  56.00 | Upper Fence:  93.50 | Outliers:  81 ( 3.8%) | Max: 169.00
engagement_rate_per_follower   | Q1:   0.84 | Median:   2.34 | Q3:   6.53 | Upper Fence:  15.05 | Outliers: 206 (10.0%) | Max: 250.00

Outlier Investigation Decision:
All high-engagement outliers are preserved. They represent authentic viral content typical of social networks.


## 8. Cleaned Dataset Validation
We perform comprehensive quality assurance tests on the processed dataset:
1. Shape and schema verification.
2. Missing values distribution.
3. Uniqueness of `post_id` (0 duplicates).
4. Negative values verification (0 negative values).
5. Categorical consistency checks.


In [8]:
print("=== VALIDATION CHECK 1: SHAPE AND SCHEMA ===")
print(f"Cleaned dataset rows: {df.shape[0]} (Expected: 2,200)")
print(f"Cleaned dataset columns: {df.shape[1]} (Expected: 20)")
assert df.shape == (2200, 20), "Shape validation failed!"

print("\n=== VALIDATION CHECK 2: DUPLICATES ===")
dup_ids = df['post_id'].duplicated().sum()
print(f"Duplicate post_ids: {dup_ids} (Expected: 0)")
assert dup_ids == 0, "Duplicate ID validation failed!"

print("\n=== VALIDATION CHECK 3: NEGATIVE NUMERIC VALUES ===")
numeric_cols = ['followers_count', 'likes', 'retweets_shares', 'comments', 'total_engagement', 'engagement_rate_per_follower']
for col in numeric_cols:
    neg_count = (df[col].dropna() < 0).sum()
    print(f"Negative values in '{col}': {neg_count} (Expected: 0)")
    assert neg_count == 0, f"Found negative values in {col}!"

print("\n=== VALIDATION CHECK 4: CATEGORICAL INTEGRITY ===")
print(f"Platforms: {sorted(df['platform'].unique().tolist())} (Expected: 5)")
assert len(df['platform'].unique()) == 5, "Platform count validation failed!"
print(f"Countries: {sorted(df['country'].unique().tolist())} (Expected: 12 + 'Unknown')")
assert len(df['country'].unique()) == 13, "Country count validation failed!"

print("\n=== VALIDATION CHECK 5: MISSING VALUE AUDIT ===")
missing_summary = pd.DataFrame({
    'Missing Count': df.isnull().sum(),
    'Missing Percentage (%)': (df.isnull().sum() / len(df) * 100).round(2)
})
print(missing_summary.to_string())

print("\nALL VALIDATION CHECKS PASSED SUCCESSFULLY!")


=== VALIDATION CHECK 1: SHAPE AND SCHEMA ===
Cleaned dataset rows: 2200 (Expected: 2,200)
Cleaned dataset columns: 20 (Expected: 20)

=== VALIDATION CHECK 2: DUPLICATES ===
Duplicate post_ids: 0 (Expected: 0)

=== VALIDATION CHECK 3: NEGATIVE NUMERIC VALUES ===
Negative values in 'followers_count': 0 (Expected: 0)
Negative values in 'likes': 0 (Expected: 0)
Negative values in 'retweets_shares': 0 (Expected: 0)
Negative values in 'comments': 0 (Expected: 0)
Negative values in 'total_engagement': 0 (Expected: 0)
Negative values in 'engagement_rate_per_follower': 0 (Expected: 0)

=== VALIDATION CHECK 4: CATEGORICAL INTEGRITY ===
Platforms: ['Facebook', 'Instagram', 'LinkedIn', 'TikTok', 'Twitter'] (Expected: 5)
Countries: ['Australia', 'Brazil', 'Canada', 'France', 'Germany', 'India', 'Japan', 'Mexico', 'South Africa', 'UAE', 'UK', 'USA', 'Unknown'] (Expected: 12 + 'Unknown')

=== VALIDATION CHECK 5: MISSING VALUE AUDIT ===
                              Missing Count  Missing Percentage (

## 9. Processed Dataset Export
The cleaned and validated dataset is exported to `data/processed/social_media_engagement_cleaned.csv`.
We verify that the exported CSV can be read back cleanly with identical dimensions.


In [9]:
# Ensure processed folder exists
os.makedirs(processed_dir, exist_ok=True)

# Export cleaned dataset
df.to_csv(processed_csv_path, index=False)
print(f"Cleaned dataset exported to: {processed_csv_path}")
print(f"Exported file size: {os.path.getsize(processed_csv_path):,} bytes")

# Verification read-back
df_verify = pd.read_csv(processed_csv_path)
print(f"\nVerification read-back:")
print(f"  Rows loaded:    {df_verify.shape[0]}")
print(f"  Columns loaded: {df_verify.shape[1]}")
assert df_verify.shape == (2200, 20), "Export verification shape mismatch!"
print("Export successfully verified and ready for Milestone 3 (Exploratory Data Visualization).")


Cleaned dataset exported to: data\processed\social_media_engagement_cleaned.csv
Exported file size: 433,648 bytes

Verification read-back:
  Rows loaded:    2200
  Columns loaded: 20
Export successfully verified and ready for Milestone 3 (Exploratory Data Visualization).
